# 01 — Data Download

Downloads all automated data sources (yfinance, FRED) and provides instructions for manual downloads (CBOE, Kaggle).

**Outputs:** Raw CSV files in `data/raw/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.data.download import download_yfinance, download_fred, load_cboe_files, load_options_data

## 1. yfinance: SPY, VIX, VVIX, HYG, LQD, DXY

In [ ]:
yf_data = download_yfinance(start='2010-01-01', end='2024-01-01')

for name, df in yf_data.items():
    print(f'{name}: {df.shape}, {df.index[0].date()} to {df.index[-1].date()}')

## 2. FRED: Treasury Yields (DGS2, DGS10)

Set your FRED API key as an environment variable:
```bash
export FRED_API_KEY='your_key_here'
```
Or pass it directly below.

In [ ]:
# Option 1: Use environment variable
treasury = download_fred(start='2010-01-01', end='2024-01-01')

# Option 2: Pass key directly (uncomment)
# treasury = download_fred(start='2010-01-01', end='2024-01-01', api_key='YOUR_KEY')

print(f'Treasury: {treasury.shape}')
treasury.head()

## 3. CBOE Manual Downloads

Download these files from [CBOE Historical Data](https://www.cboe.com/us/options/market_statistics/historical_data/) and place them in `data/raw/`:

| File | Save as |
|------|--------|
| VIX9D historical CSV | `data/raw/vix9d.csv` |
| VIX3M historical CSV | `data/raw/vix3m.csv` |
| VIX6M historical CSV | `data/raw/vix6m.csv` |
| Put/Call Ratio CSV | `data/raw/put_call_ratio.csv` |

In [ ]:
cboe = load_cboe_files()
for name, series in cboe.items():
    if series is not None:
        print(f'{name}: {len(series)} rows')
    else:
        print(f'{name}: NOT FOUND — download from CBOE')

## 4. Kaggle: SPY Options Data

Download from [Kaggle](https://www.kaggle.com/datasets/shubhamcodez/s-and-p-500-daily-options-data-2010-2023) and save as `data/raw/spy_options.csv`.

In [ ]:
try:
    opts = load_options_data()
    print(f'Options data: {opts.shape}')
    print(f'Date range: {opts["date"].min()} to {opts["date"].max()}')
    print(f'Columns: {list(opts.columns)}')
except FileNotFoundError as e:
    print(e)

## Verification

Check all files are present before proceeding to preprocessing.

In [ ]:
import glob

raw_dir = os.path.join('..', 'data', 'raw')
files = sorted(glob.glob(os.path.join(raw_dir, '*.csv')))
print(f'Files in data/raw/ ({len(files)}):')
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f):30s}  {size_mb:>8.2f} MB')